# Example notebook

In [1]:
#reqs
import pandas as pd
import geopandas as gpd
import urllib
import json
import zzproxies
from zzproxies.core import registry
print("zzproxies version: ", zzproxies.__version__)

zzproxies version:  0.9.0


In [9]:
# helpers
def get_location_context(address: str, padding: float = 0.005):
    """
    Translates an address into a BBOX and an ISO Country Code.
    """
    url = f"https://nominatim.openstreetmap.org/search?q={urllib.parse.quote(address)}&format=json&addressdetails=1&limit=1"
    headers = {"User-Agent": "zzproxies-research-lab"}
    
    req = urllib.request.Request(url, headers=headers)
    with urllib.request.urlopen(req) as response:
        data = json.loads(response.read().decode())
        
    if not data:
        raise ValueError(f"Address '{address}' not found.")
        
    loc = data[0]
    lat, lon = float(loc["lat"]), float(loc["lon"])
    country_code = loc.get("address", {}).get("country_code", "").upper()
    
    # Generate BBOX [xmin, ymin, xmax, ymax]
    bbox = [lon - padding, lat - padding, lon + padding, lat + padding]
    
    return bbox, country_code

import lonboard
from lonboard import SolidPolygonLayer, Map
import palettable
import numpy as np

def plot_building_choropleth(gdf, color_col='building_type'):
    # 1. Get unique building types and pick a palette
    unique_types = gdf[color_col].unique().tolist()
    # Tableau_10 is great for distinct categorical data
    palette = palettable.tableau.Tableau_10.colors 
    
    # 2. Create a lookup dictionary { 'Residential': [r, g, b], ... }
    color_lookup = {
        b_type: palette[i % len(palette)] 
        for i, b_type in enumerate(unique_types)
    }
    
    # 3. Apply colors to the GDF (Lonboard needs a list of [R, G, B] for each row)
    # We use .map() for speed and cast to a list of lists
    gdf['color'] = gdf[color_col].map(color_lookup)
    
    # 4. Handle any potential null colors (safety check)
    gdf['color'] = gdf['color'].apply(lambda x: x if x is not None else [200, 200, 200])

    # 5. Create the Layer
    layer = SolidPolygonLayer.from_geopandas(
        gdf,
        get_fill_color=np.array(gdf['color'].tolist(), dtype=np.uint8),
        opacity=0.8,
        pickable=True,
        auto_highlight=True
    )
    m = Map(layers=[layer])
    return m


In [3]:
# The research site
ADDRESS = "Otakaari 1, Espoo, Finland" 

# Geocode the intent
bbox, country_code = get_location_context(ADDRESS, padding=0.005)

print(f"📍 Research Site: {ADDRESS}")
print(f"🌍 Country Code: {country_code}")
print(f"📦 BBOX: {bbox}")

📍 Research Site: Otakaari 1, Espoo, Finland
🌍 Country Code: FI
📦 BBOX: [24.824994800000002, 60.1812949, 24.8349948, 60.1912949]


In [11]:
# Trigger the full pipeline

# Options: "upfront_gwp_supply", "activity_gwp_pcf", "slf_proxy"
PROXY_NAME = "slf_proxy" #"upfront_gwp_supply" #"activity_gwp_pcf"

results = None
try:
    results = registry.execute_pipeline(
        proxy_name=PROXY_NAME,
        bbox=bbox,
        country_code=country_code
    )
    
    metadata = registry.get_metadata(PROXY_NAME)
    status = metadata["status"]
    
    print(f"Execution Successful!")
    print(f"Proxy: {status.name} (v{status.version})")
    print(f"Methodology: {status.description}")
    print(f"---")
    print(f"✅ Processed {len(results)} buildings in the district.")

except PermissionError as e:
    print(f"❌ Geographic Error: {e}")
except Exception as e:
    print(f"❌ Pipeline Error: {e}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Execution Successful!
Proxy: Situated Lifestyle Footprint (v0.9.0)
Methodology: Measures lifestyle GWP using the Situated Lifestyle Footprint (SLF) factor.
---
✅ Processed 69 buildings in the district.


In [12]:
#plot
df = pd.DataFrame(results)
gdf = gpd.GeoDataFrame(
        df, 
        geometry=gpd.GeoSeries.from_wkt(df['wkt']), 
        crs="EPSG:4326"
    )
m = plot_building_choropleth(gdf,color_col='GWPL')
m

In [6]:
#
print(df['building_type'].value_counts())
df.head(3)


building_type
public              46
apartment-condo     11
commercial           6
industrial           4
one-family-house     1
other                1
Name: count, dtype: int64


,id,building_type,num_floors,GFA,NFA_by_industry,wkt,proxy_value,proxy_unit,GWPL
0,92279aef-ced4-4c87-b130-8c3fa14921b0,apartment-condo,4,4460.0,"{'Retail & Food': 0.0, 'Accommodation': 0.0, '...","POLYGON ((24.8338147 60.1900111, 24.8340197 60...",3480.0,t_CO2e_upfront,L4: Critical Carbon Debt
1,01ee3416-ee82-49e9-948b-90be7c9f7962,public,1,60.0,"{'Retail & Food': 0.0, 'Accommodation': 0.0, '...","POLYGON ((24.831603 60.1815186, 24.8316175 60....",50.0,t_CO2e_upfront,L0: Low Carbon Debt
2,3ec4c289-0e1e-42c3-b0f7-eae0706d25ad,public,1,0.0,"{'Retail & Food': 0.0, 'Accommodation': 0.0, '...","POLYGON ((24.8311713 60.184431, 24.8311636 60....",0.0,t_CO2e_upfront,L0: Low Carbon Debt


### Comparisons

In [13]:
# -- comparison ---
from zzproxies import UrbanImpactReport, compare_reports

bbox_a, country_code = get_location_context("Mankkaa, Espoo", padding=0.003)
bbox_b, country_code = get_location_context("Malmi, Helsinki", padding=0.003)

# 1. Pipeline Execution
res_site_a = registry.execute_pipeline(PROXY_NAME, bbox_a, "FI")
res_site_b = registry.execute_pipeline(PROXY_NAME, bbox_b, "FI")

# 2. Report Creation
report_a = UrbanImpactReport("Mankkaa", res_site_a)
report_b = UrbanImpactReport("Malmi", res_site_b)

# 3. Comparison as DataFrame
comparison_table = pd.DataFrame(compare_reports([report_a, report_b]))

print("Impact-Verified Zoning Comparison ready!")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Impact-Verified Zoning Comparison ready!


In [14]:
comparison_table

,district_name,total_impact,unit,gwpl_distribution,critical_assets_L4,high_risk_assets_L3,primary_methodology
0,Mankkaa,4580.0,t_CO2e_lifestyle_annual,{'L0': 131},0,0,N/A
1,Malmi,16200.0,t_CO2e_lifestyle_annual,"{'L3': 4, 'L1': 11, 'L0': 36, 'L4': 1, 'L2': 3}",1,4,N/A
